# Fire Type Classification - Week 3 Project
This notebook builds and saves a fire classification model using MODIS satellite data.

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib


## Load and Inspect Dataset

In [ ]:

# Replace with actual dataset path or use sample data for simulation
data = pd.read_csv("MODIS_Fire_Data.csv")  # Ensure you have this CSV or adapt accordingly
data.head()


## Preprocess Data

In [ ]:

# Encode confidence level
confidence_map = {"low": 0, "nominal": 1, "high": 2}
data["confidence"] = data["confidence"].map(confidence_map)

# Select features and labels
X = data[["brightness", "bright_t31", "frp", "scan", "track", "confidence"]]
y = data["type"]  # assuming column 'type' has labels: 0, 2, 3

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Train Model

In [ ]:

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)


## Evaluate Model

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Save Model and Scaler

In [ ]:

joblib.dump(model, "best_fire_detection_model.pkl")
joblib.dump(scaler, "scaler.pkl")


## Streamlit App Code (Deploy in `app.py`)

In [ ]:

import streamlit as st
import numpy as np
import joblib

# Load model and scaler
model = joblib.load("best_fire_detection_model.pkl")
scaler = joblib.load("scaler.pkl")

st.set_page_config(page_title="Fire Type Classifier", layout="centered")
st.title("Fire Type Classification")
st.markdown("Predict fire type based on MODIS satellite readings.")

brightness = st.number_input("Brightness", value=300.0)
bright_t31 = st.number_input("Brightness T31", value=290.0)
frp = st.number_input("Fire Radiative Power (FRP)", value=15.0)
scan = st.number_input("Scan", value=1.0)
track = st.number_input("Track", value=1.0)
confidence = st.selectbox("Confidence Level", ["low", "nominal", "high"])

confidence_map = {"low": 0, "nominal": 1, "high": 2}
confidence_val = confidence_map[confidence]

input_data = np.array([[brightness, bright_t31, frp, scan, track, confidence_val]])
scaled_input = scaler.transform(input_data)

if st.button("Predict Fire Type"):
    prediction = model.predict(scaled_input)[0]
    fire_types = {0: "Vegetation Fire", 2: "Other Static Land Source", 3: "Offshore Fire"}
    result = fire_types.get(prediction, "Unknown")
    st.success(f"**Predicted Fire Type:** {result}")
